# ANN Fraud Detection on Indian Financial Transactions Dataset

This notebook loads the dataset, builds a simple artificial neural network (ANN), trains it to detect fraud, logs TensorBoard metrics, and saves the trained model in a `.pkl` file.

## 1. Install required libraries

Install TensorFlow and TensorBoard if needed. Run this cell once if the environment does not already have them.

In [2]:
# !pip install -q tensorflow tensorboard pandas scikit-learn

## 2. Import libraries and load the dataset

Read the CSV file and inspect its columns and first rows.

In [3]:
import pandas as pd
from pathlib import Path

data_path = Path('indian_financial_transactions (2).csv')
df = pd.read_csv(data_path)

print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head(5)

Dataset shape: (210300, 22)
Columns: ['transaction_id', 'customer_id', 'customer_age', 'customer_gender', 'customer_city', 'customer_state', 'account_type', 'annual_income', 'credit_score', 'account_tenure_months', 'transaction_date', 'transaction_hour', 'day_of_week', 'transaction_type', 'transaction_category', 'merchant_name', 'payment_method', 'device_type', 'amount', 'is_international', 'account_balance_after', 'is_fraud']


,transaction_id,customer_id,customer_age,customer_gender,customer_city,customer_state,account_type,annual_income,credit_score,account_tenure_months,...,day_of_week,transaction_type,transaction_category,merchant_name,payment_method,device_type,amount,is_international,account_balance_after,is_fraud
0,TXN0027758,CUST05844,18,Female,Kolkata,West Bengal,Salary,516000.0,542.0,39,...,Saturday,Debit,Rent,NoBroker,UPI,ATM,16994.34,0,35612.55,0
1,TXN0148920,CUST01600,55,Male,Kochi,Kerala,Salary,472000.0,585.0,29,...,Sunday,Debit,Dining,Swiggy,Debit Card,Mobile App,306.97,0,48737.05,0
2,TXN0002294,CUST01985,51,Male,Ahmedabad,Gujarat,Salary,618000.0,545.0,57,...,Wednesday,Debit,Shopping,Flipkart,Wallet,Mobile App,1846.87,0,92627.34,0
3,TXN0006598,CUST00052,28,Male,Kolkata,West Bengal,Salary,295000.0,631.0,78,...,Friday,Debit,Travel,Ola,Net Banking,Mobile App,1488.24,0,38506.30,0
4,TXN0051387,CUST01610,24,Female,Lucknow,Uttar Pradesh,Current,206000.0,550.0,68,...,Saturday,Debit,Entertainment,Hotstar,Net Banking,Mobile App,240.07,0,47738.79,0


## 3. Preprocess the data

Select useful columns, encode categorical values, and prepare feature matrices for training.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Choose a subset of columns for simplicity
selected_columns = [
    'customer_age', 'customer_gender', 'account_type', 'annual_income',
    'credit_score', 'account_tenure_months', 'transaction_hour', 'day_of_week',
    'transaction_type', 'transaction_category', 'payment_method', 'device_type',
    'amount', 'is_international', 'account_balance_after'
]

df_small = df[selected_columns + ['is_fraud']].copy()

# Fill missing numeric values with the column median
numeric_cols = ['customer_age', 'annual_income', 'credit_score', 'account_tenure_months', 'transaction_hour', 'amount', 'account_balance_after']
for col in numeric_cols:
    df_small[col] = pd.to_numeric(df_small[col], errors='coerce')
    df_small[col] = df_small[col].fillna(df_small[col].median())

# Convert categorical columns using one-hot encoding
categorical_cols = ['customer_gender', 'account_type', 'day_of_week', 'transaction_type', 'transaction_category', 'payment_method', 'device_type']
df_encoded = pd.get_dummies(df_small, columns=categorical_cols, drop_first=True)

# Separate features and label
X = df_encoded.drop('is_fraud', axis=1)
y = df_encoded['is_fraud']

print('Final feature shape:', X.shape)
print('Fraud label counts:')
print(y.value_counts())

Final feature shape: (210300, 42)
Fraud label counts:
is_fraud
0    207603
1      2697
Name: count, dtype: int64


## 4. Split and scale the data

Split the dataset into training and test sets and scale numeric values for better neural network training.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train[X_train.columns] = scaler.fit_transform(X_train[X_train.columns])
X_test[X_test.columns] = scaler.transform(X_test[X_test.columns])

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)

Training shape: (168240, 42)
Test shape: (42060, 42)


## 5. Build a simple ANN model

Create a Keras sequential model with two hidden layers and a sigmoid output for binary classification.

In [6]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


/opt/anaconda3/envs/tf_env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,865 (19.00 KB)

 Trainable params: 4,865 (19.00 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Train with TensorBoard logging

Train the model and log metrics to TensorBoard for visualization.

In [19]:
import datetime
from tensorflow.keras.callbacks import TensorBoard

log_dir = 'logs/fit/' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S')
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=256,
    callbacks=[tensorboard_callback],
    verbose=1
)

Epoch 1/10
526/526 ━━━━━━━━━━━━━━━━━━━━ 0s 716us/step - accuracy: 0.9875 - loss: 0.0551 - val_accuracy: 0.9874 - val_loss: 0.0691
Epoch 2/10
526/526 ━━━━━━━━━━━━━━━━━━━━ 0s 631us/step - accuracy: 0.9875 - loss: 0.0545 - val_accuracy: 0.9873 - val_loss: 0.0690
Epoch 3/10
526/526 ━━━━━━━━━━━━━━━━━━━━ 0s 625us/step - accuracy: 0.9873 - loss: 0.0549 - val_accuracy: 0.9874 - val_loss: 0.0694
Epoch 4/10
526/526 ━━━━━━━━━━━━━━━━━━━━ 0s 612us/step - accuracy: 0.9873 - loss: 0.0551 - val_accuracy: 0.9872 - val_loss: 0.0693
Epoch 5/10
526/526 ━━━━━━━━━━━━━━━━━━━━ 0s 618us/step - accuracy: 0.9874 - loss: 0.0548 - val_accuracy: 0.9873 - val_loss: 0.0692
Epoch 6/10
526/526 ━━━━━━━━━━━━━━━━━━━━ 0s 637us/step - accuracy: 0.9874 - loss: 0.0549 - val_accuracy: 0.9875 - val_loss: 0.0697
Epoch 7/10
526/526 ━━━━━━━━━━━━━━━━━━━━ 0s 622us/step - accuracy: 0.9874 - loss: 0.0546 - val_accuracy: 0.9873 - val_loss: 0.0696
Epoch 8/10
526/526 ━━━━━━━━━━━━━━━━━━━━ 0s 624us/step - accuracy: 0.9875 - loss: 0.0540 - 

## 7. Evaluate the model

Check accuracy and loss on the test dataset.

In [8]:
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f'Test loss: {test_loss:.4f}')
print(f'Test accuracy: {test_accuracy:.4f}')

1315/1315 - 0s - 237us/step - accuracy: 0.9872 - loss: 0.0646
Test loss: 0.0646
Test accuracy: 0.9872


## 8. Save the model to a `.pkl` file

Save the model architecture and weights in a pickle file so the trained model can be loaded later.

In [9]:
import pickle

model_data = {
    'model_config': model.to_json(),
    'model_weights': model.get_weights(),
    'scaler_mean': scaler.mean_.tolist(),
    'scaler_scale': scaler.scale_.tolist(),
    'feature_columns': X_train.columns.tolist(),
}

with open('ann_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print('Saved model to ann_model.pkl')

Saved model to ann_model.pkl


## 9. Load the model from `.pkl` (optional)

Demonstrate how to restore the model structure and weights from the pickle file.

In [10]:
from tensorflow.keras.models import model_from_json

with open('ann_model.pkl', 'rb') as f:
    loaded_data = pickle.load(f)

loaded_model = model_from_json(loaded_data['model_config'])
loaded_model.set_weights(loaded_data['model_weights'])
loaded_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

print('Loaded model from ann_model.pkl')

Loaded model from ann_model.pkl


## 10. Launch TensorBoard

Run this cell to open TensorBoard inside the notebook and view the training metrics.

In [11]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

Reusing TensorBoard on port 6006 (pid 2398), started 0:44:24 ago. (Use '!kill 2398' to kill it.)